In [6]:
import numpy as np
import pandas as pd
import re
import json
import os
import matplotlib.pyplot as plt
import seaborn as sns

In [7]:
from PIL import Image

# Voxel kopuruen analisia

In [8]:
VOXELS_ROOT_DIR = '/home/zchen002/TFG/voxels'
TRAIN_JSON = '/home/zchen002/TFG/CLEVR_images/scenes/CLEVR_train_scenes.json'
VALID_JSON = '/home/zchen002/TFG/CLEVR_images/scenes/CLEVR_val_scenes.json'
SPLIT_JSON = '/home/zchen002/TFG/kodea/data_split.json'

In [9]:
def calculate_dataset_metrics():
    with open(SPLIT_JSON, 'r') as f:
        data_split = json.load(f)
    
    train_indices = set(data_split.get('train', []))
    valid_indices = set(data_split.get('val', []))
    
    metrics_data = {
        'TRAIN': {'ids': [], 'filled': [], 'empty': []},
        'VALID': {'ids': [], 'filled': [], 'empty': []},
        'TEST': {'ids': [], 'filled': [], 'empty': []}
    }
    
    train_folder_path = os.path.join(VOXELS_ROOT_DIR, 'train')
    if os.path.exists(train_folder_path):
        for filename in os.listdir(train_folder_path):
            if filename.endswith('.npy'):
                voxel_path = os.path.join(train_folder_path, filename)
                try:
                    parts = filename.replace('.npy', '').split('_')
                    index = int(parts[-1])
                except ValueError:
                    continue
                
                if index in train_indices:
                    current_set = 'TRAIN'
                elif index in valid_indices:
                    current_set = 'VALID'
                else:
                    continue
                
                voxels = np.load(voxel_path)
                filled = np.sum(voxels > 0)
                empty = np.sum(voxels == 0)
                
                metrics_data[current_set]['ids'].append(filename)
                metrics_data[current_set]['filled'].append(filled)
                metrics_data[current_set]['empty'].append(empty)

    test_folder_path = os.path.join(VOXELS_ROOT_DIR, 'test')
    if os.path.exists(test_folder_path):
        for filename in os.listdir(test_folder_path):
            if filename.endswith('.npy'):
                voxel_path = os.path.join(test_folder_path, filename)
                voxels = np.load(voxel_path)
                filled = np.sum(voxels > 0)
                empty = np.sum(voxels == 0)
                
                metrics_data['TEST']['ids'].append(filename)
                metrics_data['TEST']['filled'].append(filled)
                metrics_data['TEST']['empty'].append(empty)

    for set_name, data in metrics_data.items():
        print(f"=========================================\n  METRICS FOR {set_name} DATASET\n=========================================")
        ids_list = data['ids']
        filled_list = data['filled']
        empty_list = data['empty']
        
        if not ids_list:
            print(f"No files processed for {set_name}.\n")
            continue
            
        filled_array = np.array(filled_list)
        empty_array = np.array(empty_list)
        
        min_filled_idx = np.argmin(filled_array)
        max_filled_idx = np.argmax(filled_array)
        
        min_empty_idx = np.argmin(empty_array)
        max_empty_idx = np.argmax(empty_array)
        
        print(f"Total processed scenes: {len(ids_list)}")
        print(f"FILLED voxels -> Mean: {np.mean(filled_array):.2f} | Std dev: {np.std(filled_array):.2f}")
        print(f"  - Min: {filled_array[min_filled_idx]} (ID: {ids_list[min_filled_idx]})")
        print(f"  - Max: {filled_array[max_filled_idx]} (ID: {ids_list[max_filled_idx]})")
        
        print(f"EMPTY voxels  -> Mean: {np.mean(empty_array):.2f} | Std dev: {np.std(empty_array):.2f}")
        print(f"  - Min: {empty_array[min_empty_idx]} (ID: {ids_list[min_empty_idx]})")
        print(f"  - Max: {empty_array[max_empty_idx]} (ID: {ids_list[max_empty_idx]})")
        print("\n")

In [10]:
if __name__ == "__main__":
    calculate_dataset_metrics()

  METRICS FOR TRAIN DATASET
Total processed scenes: 59500
FILLED voxels -> Mean: 20584.56 | Std dev: 9607.26
  - Min: 1485 (ID: scene_067506.npy)
  - Max: 64461 (ID: scene_041924.npy)
EMPTY voxels  -> Mean: 241559.44 | Std dev: 9607.26
  - Min: 197683 (ID: scene_041924.npy)
  - Max: 260659 (ID: scene_067506.npy)


  METRICS FOR VALID DATASET
Total processed scenes: 10500
FILLED voxels -> Mean: 20730.23 | Std dev: 9680.09
  - Min: 1484 (ID: scene_018284.npy)
  - Max: 60877 (ID: scene_001165.npy)
EMPTY voxels  -> Mean: 241413.77 | Std dev: 9680.09
  - Min: 201267 (ID: scene_001165.npy)
  - Max: 260660 (ID: scene_018284.npy)


  METRICS FOR TEST DATASET
Total processed scenes: 15000
FILLED voxels -> Mean: 20456.47 | Std dev: 9518.33
  - Min: 1493 (ID: scene_001322.npy)
  - Max: 57109 (ID: scene_004064.npy)
EMPTY voxels  -> Mean: 241687.53 | Std dev: 9518.33
  - Min: 205035 (ID: scene_004064.npy)
  - Max: 260651 (ID: scene_001322.npy)




In [ ]:
import os
import matplotlib.pyplot as plt

# 1. Definición de los datos extraídos de tus métricas
datasets_metrics = {
    'TRAIN': {'filled': 20584.56, 'empty': 241559.44},
    'VALID': {'filled': 20730.23, 'empty': 241413.77},
    'TEST': {'filled': 20456.47, 'empty': 241687.53}
}

def create_pie_charts(metrics, output_dir='plots'):
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    labels = ['Okuaptuta (1)', 'Hutsik (0)']
    colors = ['#ff9999', '#66b3ff']
    explode = (0.1, 0)  

    for set_name, values in metrics.items():
        sizes = [values['filled'], values['empty']]
        
        plt.figure(figsize=(6, 6))
        
        plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%', shadow=True, startangle=140)
        
        plt.title(f'Voxel Banaketa - {set_name} multzoa\n(Guztira: {sum(sizes):,.0f} voxels)')
        
        file_path = os.path.join(output_dir, f'{set_name.lower()}_voxel_distribution.png')
        plt.savefig(file_path, bbox_inches='tight', dpi=300)
        plt.close()
        
        print(f"Pie chart saved for {set_name} dataset at: {file_path}")

if __name__ == "__main__":
    create_pie_charts(datasets_metrics)

Pie chart saved for TRAIN dataset at: plots/train_voxel_distribution.png
Pie chart saved for VALID dataset at: plots/valid_voxel_distribution.png
Pie chart saved for TEST dataset at: plots/test_voxel_distribution.png
